# Vendor Invoice Intelligence System
**Freight Cost Prediction & Invoice Risk Flagging**

**Do parts:**
1. Freight Cost Prediction (Regression) — `vendor_invoice` table, target = `Freight`
2. Invoice Manual-Approval / Risk Flagging (Classification) — invoice-level features via SQl

In [ ]:
import sqlite3
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    make_scorer, f1_score,
)

import warnings
warnings.filterwarnings("ignore")

## 1. Database load (inventory.db)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#inventory.db
DB_PATH = "/content/drive/MyDrive/inventory.db"

In [ ]:
import os
assert os.path.exists(DB_PATH), f"inventory.db nahi mili: {DB_PATH}  -> path check karo"
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("select name from sqlite_master where type='table'", conn)
tables

In [ ]:
for table in tables['name']:
    print(f'Table name: {table}')
    display(pd.read_sql(f"select * from {table} limit 5", conn))

---
# PART 1 — Freight Cost Prediction (Regression)
**Objective:** Vendor invoice ka freight cost quantity/dollars se predict karna — forecasting, budgeting aur vendor negotiation ke liye.

In [ ]:
vendor_df = pd.read_sql_query("select * from vendor_invoice", conn)
vendor_df.head()

In [ ]:
vendor_df[['Quantity','Freight','Dollars']].corr()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(vendor_df['Quantity'], vendor_df['Freight'], c='#c0392b', s=10, label='Quantity')
plt.scatter(vendor_df['Dollars'], vendor_df['Freight'], c='#4a0d67', s=10, label='Dollars')
plt.ylabel('Freight')
plt.legend()
plt.show()

In [ ]:
vendor_df['freight_per_unit'] = vendor_df['Freight'] / vendor_df['Quantity']

high_quantity = vendor_df['Quantity'].quantile(0.75)
low_quantity  = vendor_df['Quantity'].quantile(0.25)
print("high_quantity:", high_quantity, "| low_quantity:", low_quantity)

print("freight/unit (low qty) :", vendor_df.loc[vendor_df['Quantity'] < low_quantity,  'freight_per_unit'].mean())
print("freight/unit (high qty):", vendor_df.loc[vendor_df['Quantity'] > high_quantity, 'freight_per_unit'].mean())
vendor_df.describe().round()

In [ ]:
# ---------- data_preprocessing.py ----------
def load_vendor_invoice_data(db_path: str):
    """Load vendor invoice data from SQLite database."""
    conn = sqlite3.connect(db_path)
    query = "SELECT * FROM vendor_invoice"
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

def prepare_features(df: pd.DataFrame):
    """Select features and target variable."""
    X = df[["Dollars"]]
    y = df["Freight"]
    return X, y

def split_data(X, y, test_size=0.2, random_state=42):
    """Split dataset into train and test sets."""
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

In [ ]:
# ---------- model_evaluation.py ----------
def train_linear_regression(X_train, y_train):
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

def train_decision_tree(X_train, y_train, max_depth=5):
    model = DecisionTreeRegressor(
        max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)
    return model

def train_random_forest(X_train, y_train, max_depth=6):
    model = RandomForestRegressor(
        max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)
    return model

def evaluate_regression_model(model, X_test, y_test, model_name: str) -> dict:
    """Evaluate regression model and return metrics."""
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds) * 100

    print(f"\n{model_name} Performance:")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.2f}%")

    return {
        "model_name": model_name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

In [ ]:
# ---------- train.py (main) ----------
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

# Load data
df_freight = load_vendor_invoice_data(DB_PATH)

# Prepare data
X, y = prepare_features(df_freight)
X_train, X_test, y_train, y_test = split_data(X, y)

# Train models
lr_model = train_linear_regression(X_train, y_train)
dt_model = train_decision_tree(X_train, y_train)
rf_model = train_random_forest(X_train, y_train)

# Evaluate models
results = []
results.append(evaluate_regression_model(lr_model, X_test, y_test, "Linear Regression"))
results.append(evaluate_regression_model(dt_model, X_test, y_test, "Decision Tree Regression"))
results.append(evaluate_regression_model(rf_model, X_test, y_test, "Random Forest Regression"))

# Select best model (lowest MAE)
best_model_info = min(results, key=lambda x: x["mae"])
best_model_name = best_model_info["model_name"]

best_model = {
    "Linear Regression": lr_model,
    "Decision Tree Regression": dt_model,
    "Random Forest Regression": rf_model
}[best_model_name]

# Save best model
model_path = model_dir / "predict_freight_model.pkl"
joblib.dump(best_model, model_path)

print(f"\nBest model saved: {best_model_name}")
print(f"Model path: {model_path}")

---
# PART 2 — Invoice Manual-Approval / Risk Flagging (Classification)

In [ ]:
purchase_agg_df = pd.read_sql_query("""
select
    p.PONumber,
    count(distinct p.Brand) as total_brands,
    sum(p.Quantity) as total_item_quantity,
    sum(p.Dollars) as total_item_dollars,
    avg(julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_receiving_delay
from purchases p
group by p.PONumber
""", conn)
purchase_agg_df.shape

In [ ]:
df = pd.read_sql_query("""
WITH purchase_agg AS (
    SELECT
        p.PONumber,
        COUNT(DISTINCT p.Brand) AS total_brands,
        SUM(p.Quantity) AS total_item_quantity,
        SUM(p.Dollars) AS total_item_dollars,
        AVG(julianday(p.ReceivingDate) - julianday(p.PODate)) AS avg_receiving_delay
    FROM purchases p
    GROUP BY p.PONumber
)

SELECT
    vi.PONumber,
    vi.Quantity AS invoice_quantity,
    vi.Dollars AS invoice_dollars,
    vi.Freight,
    (julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
    (julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
    pa.total_brands,
    pa.total_item_quantity,
    pa.total_item_dollars,
    pa.avg_receiving_delay

FROM vendor_invoice vi
LEFT JOIN purchase_agg pa
    ON vi.PONumber = pa.PONumber

""", conn)
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
def create_invoice_risk_label(row):

    # Invoice total mismatch with item-level total
    if (abs(row["invoice_dollars"] - row["total_item_dollars"]) > 5):
        return 1

    # Abnormally high receiving delay
    if row["avg_receiving_delay"] > 10:
        return 1

    return 0

df["flag_invoice"] = df.apply(create_invoice_risk_label, axis=1)
df["flag_invoice"].value_counts()

### Statistical test (t-test) — flagged vs normal invoices

In [ ]:
flagged = df[df['flag_invoice']==1]
normal = df[df['flag_invoice']==0]

significant_features = []
non_significant_features = []
results = []

metrics = ['invoice_quantity', 'invoice_dollars', 'Freight',
           'days_po_to_invoice', 'days_to_pay', 'total_brands',
           'total_item_quantity', 'total_item_dollars', 'avg_receiving_delay']

for metric in metrics:
    flagged_mean = flagged[metric].mean()
    normal_mean = normal[metric].mean()

    t_stat, p_value = ttest_ind(
        flagged[metric].dropna(),
        normal[metric].dropna(),
        equal_var=False
    )

    if p_value < 0.05:
        significant_features.append(metric)
        results.append({
        "metric": metric,
        "flagged_mean": flagged_mean.round(2),
        "normal_mean": normal_mean.round(2),
        "p_value": p_value.round(3)
        })
    else:
        non_significant_features.append(metric)

print("Non-significant:", non_significant_features)
pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df[metrics].corr(), annot=True, fmt=".2g", cmap="magma")
plt.show()

### Features → Train/Test split → Scaling → Models

In [ ]:
X = df[['invoice_quantity','invoice_dollars','Freight','total_item_quantity','total_item_dollars']]
y = df['flag_invoice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X.describe().round()

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds)

    print(f"\n{model_name} Performance:")
    print(f"Accuracy : {accuracy:.2f}")
    print("Classification Report :")
    print(report)

In [ ]:
model1 = LogisticRegression(random_state=42)
model1.fit(X_train_scaled, y_train)

model2 = DecisionTreeClassifier(random_state = 42)
model2.fit(X_train_scaled, y_train)

model3 = RandomForestClassifier(random_state = 42)
model3.fit(X_train_scaled, y_train)

evaluate_model(model1, X_test_scaled, y_test, 'Logistic Regression')
evaluate_model(model2, X_test_scaled, y_test, 'Decision Tree Classifier')
evaluate_model(model3, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model3.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

### Hyperparameter tuning — Random Forest + GridSearchCV (F1 score)

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100,200, 300],
    "max_depth": [None, 4, 5, 6],
    "min_samples_split": [2, 3, 5],
    "min_samples_leaf": [1, 2, 5],
    "criterion":['gini','entropy']
}
scorer = make_scorer(f1_score)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)
evaluate_model(grid_search, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
print("Best params:", grid_search.best_params_)
confusion_matrix(y_test, grid_search.predict(X_test_scaled))

### Model save + inference

In [ ]:
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

joblib.dump(grid_search.best_estimator_, model_dir / "predict_flag_invoice_model.pkl")
joblib.dump(scaler, model_dir / "scaler.pkl")
print("Saved:", [p.name for p in model_dir.iterdir()])

In [ ]:
def predict_freight_cost(input_data: dict) -> pd.DataFrame:
    model = joblib.load(model_dir / "predict_freight_model.pkl")
    input_df = pd.DataFrame(input_data)
    input_df["Predicted_Freight"] = model.predict(input_df).round(2)
    return input_df

def predict_invoice_flag(input_data: dict) -> pd.DataFrame:
    model = joblib.load(model_dir / "predict_flag_invoice_model.pkl")
    sc = joblib.load(model_dir / "scaler.pkl")
    input_df = pd.DataFrame(input_data)
    input_df["Predicted_Flag"] = model.predict(sc.transform(input_df)).astype(int)
    return input_df

# ---- Freight example ----
print(predict_freight_cost({"Dollars": [18500.0]}))

# ---- Invoice flag example (app.py ke default values) ----
sample = {
    "invoice_quantity": [50],
    "invoice_dollars": [352.95],
    "Freight": [1.73],
    "total_item_quantity": [162],
    "total_item_dollars": [2476.0],
}
res = predict_invoice_flag(sample)
print(res)
print("Manual approval needed!" if bool(res["Predicted_Flag"].iloc[0]) else "Invoice looks normal.")